In [13]:
import sys, subprocess, pkgutil
def ensure(pkg):
    try:
        __import__(pkg.split("==")[0].split(">=")[0])
        print("✅", pkg)
    except Exception:
        print("… installing", pkg)
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("mistralai>=1.1.0")


✅ mistralai>=1.1.0


In [14]:
from pathlib import Path
ra = Path("research_assistant.py")
src = ra.read_text(encoding="utf-8")

fixed = src.replace(
    "from mistralai.exceptions import SDKError",
    "from mistralai.models.sdkerror import SDKError"
)
# If you previously had old client imports, you can normalize them too:
fixed = fixed.replace(
    "from mistralai.client import MistralClient",
    "from mistralai import Mistral"
)

if fixed != src:
    ra.write_text(fixed, encoding="utf-8")
    print("Patched research_assistant.py ✅")
else:
    print("Nothing to patch (already correct) ✅")


Patched research_assistant.py ✅


In [15]:
import importlib, traceback

for name in ("config","document_processor","research_assistant","research_agents"):
    try:
        importlib.invalidate_caches()
        mod = importlib.import_module(name)
        importlib.reload(mod)
        print("✅ import", name)
    except Exception:
        print("❌ import failed:", name)
        traceback.print_exc()
        raise


✅ import config
✅ import document_processor
✅ import research_assistant
✅ import research_agents


In [16]:
# Make sure the notebook is pointed at your project directory
import os, sys
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("CWD:", Path.cwd())
print("Files here:", sorted(p.name for p in PROJECT_DIR.iterdir()))


CWD: C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step
Files here: ['.env', '.ipynb_checkpoints', '.venv', 'README.md', 'Untitled.ipynb', '__pycache__', 'analyze_results.ipynb', 'config.py', 'data', 'document_processor.py', 'main.py', 'prompts', 'requirements.txt', 'research_agents.py', 'research_assistant.py', 'results', 'test_system.py', 'venv']


In [10]:
# Install anything that might be missing in the current kernel
import sys, subprocess

def ensure(pkgs):
    for p in pkgs:
        try:
            __import__(p.split("==")[0].split(">=")[0])
            print(f"✔ {p}")
        except Exception:
            print(f"… installing {p}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])

ensure([
    "mistralai>=1.1.0",
    "PyPDF2",
    "pandas",
    "numpy",
    "scikit-learn",
    "python-dotenv",
    "nltk",
])
print("All set.")


✔ mistralai>=1.1.0
✔ PyPDF2
✔ pandas
✔ numpy
… installing scikit-learn
… installing python-dotenv
✔ nltk
All set.


In [17]:
# STEP 3 — Check .env and MISTRAL_API_KEY (prints only presence, not the key)
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path(".env")
print(".env exists:", env_path.exists())
load_dotenv(env_path)  # force load here
print("Has MISTRAL_API_KEY:", bool(os.getenv("MISTRAL_API_KEY")))


.env exists: True
Has MISTRAL_API_KEY: True


In [7]:
# Make sure the notebook is pointed at your project directory
import os, sys
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("CWD:", Path.cwd())
print("Files here:", sorted(p.name for p in PROJECT_DIR.iterdir()))


CWD: C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step
Files here: ['.env', '.ipynb_checkpoints', '.venv', 'README.md', 'Untitled.ipynb', '__pycache__', 'analyze_results.ipynb', 'config.py', 'data', 'document_processor.py', 'main.py', 'prompts', 'requirements.txt', 'research_agents.py', 'research_assistant.py', 'results', 'test_system.py', 'venv']


In [18]:
from config import Config
from document_processor import DocumentProcessor
from research_assistant import ResearchGPTAssistant
from research_agents import AgentOrchestrator

cfg = Config()
dp = DocumentProcessor(cfg)

print("Data dir:", cfg.data_dir)
print("Summaries dir:", cfg.results_summaries_dir)
print("Analyses dir:", cfg.results_analyses_dir)


Data dir: C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step\data
Summaries dir: C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step\results\summaries
Analyses dir: C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step\results\analyses


In [20]:
# Load PDFs already in data/sample_papers
sample_dir = cfg.data_dir / "sample_papers"
pdfs = sorted(sample_dir.glob("*.pdf"))
print("Found PDFs:", [p.name for p in pdfs])

if not pdfs:
    raise RuntimeError(f"No PDFs in {sample_dir}. Run your downloader first (main.py) or drop PDFs here.")

# Add to processor
for p in pdfs:
    print("Adding:", p.name)
    dp.add_pdf(p)

print("Building TF-IDF index…")
dp.build_search_index()
print("Index built. chunks=", len(dp.docs), " meta=", len(dp.doc_meta))


Found PDFs: ['2509.19296v1_Lyra_Generative_3D_Scene_Reconstruction_via_Video_Diffusion_.pdf', '2509.19297v1_VolSplat_Rethinking_Feed-Forward_3D_Gaussian_Splatting_with_.pdf', '2509.19300v1_CAR-Flow_Condition-Aware_Reparameterization_Aligns_Source_an.pdf']
Adding: 2509.19296v1_Lyra_Generative_3D_Scene_Reconstruction_via_Video_Diffusion_.pdf
Adding: 2509.19297v1_VolSplat_Rethinking_Feed-Forward_3D_Gaussian_Splatting_with_.pdf
Adding: 2509.19300v1_CAR-Flow_Condition-Aware_Reparameterization_Aligns_Source_an.pdf
Building TF-IDF index…
Index built. chunks= 196  meta= 196


In [21]:
assistant = ResearchGPTAssistant(cfg, dp)
orch = AgentOrchestrator(assistant)

# Helper: collect doc_ids (they are the PDF *paths* from your index metadata)
doc_ids = sorted({m["path"] for m in dp.doc_meta})
print("Doc IDs:", doc_ids)


Doc IDs: ['C:\\Users\\Windows\\python\\lesson\\ai\\capstone folder for ai\\project_step_by_step\\data\\sample_papers\\2509.19296v1_Lyra_Generative_3D_Scene_Reconstruction_via_Video_Diffusion_.pdf', 'C:\\Users\\Windows\\python\\lesson\\ai\\capstone folder for ai\\project_step_by_step\\data\\sample_papers\\2509.19297v1_VolSplat_Rethinking_Feed-Forward_3D_Gaussian_Splatting_with_.pdf', 'C:\\Users\\Windows\\python\\lesson\\ai\\capstone folder for ai\\project_step_by_step\\data\\sample_papers\\2509.19300v1_CAR-Flow_Condition-Aware_Reparameterization_Aligns_Source_an.pdf']


In [22]:
from pathlib import Path as _P

if not doc_ids:
    raise RuntimeError("No doc_ids in index metadata.")

# Pick one doc (or loop them all)
doc_id = doc_ids[0]
summary_result = orch.route_task("summarizer", {"doc_id": doc_id})

if isinstance(summary_result, dict) and "summary" in summary_result:
    print("Summary (truncated to 800 chars):\n")
    print(summary_result["summary"][:800])

    # Save to results/summaries/
    out_path = cfg.results_summaries_dir / f"summary_{_P(doc_id).stem}.txt"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(summary_result["summary"], encoding="utf-8")
    print("\nSaved →", out_path)
else:
    print("Summarizer returned:", summary_result)


Summary (truncated to 800 chars):

Here’s a structured summary of the academic paper **"LYRA: Generative 3D Scene Reconstruction via Video Diffusion Model Self-Distillation"**:

---

### **1. Research Question/Hypothesis**
- **Core Question**:
  Can a **self-distillation framework** leverage **pre-trained video diffusion models** to generate **explicit 3D representations** (3D Gaussian Splatting, 3DGS) from **single images or monocular videos**, eliminating the need for real-world multi-view training data?
- **Hypothesis**:
  Video diffusion models implicitly encode 3D knowledge from diverse 2D videos. By distilling this knowledge into a 3DGS decoder (supervised by the video model’s RGB output), Lyra can synthesize **high-fidelity, geometrically consistent 3D/4D scenes** in a feed-forward manner, enabling real-time renderin

Saved → C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step\results\summaries\summary_2509.19296v1_Lyra_Generative_3D_Scene_Reconstructio

In [25]:
from datetime import datetime
# Define a question
query = "Compare the effectiveness of supervised vs unsupervised learning"

# Build a small context set from the index
hits = dp.find_similar_chunks(query, top_k=4)
relevant_chunks = [h[1] for h in hits]   # take only the text of each chunk

# Run CoT-style prompting (your Step-6 method returns final answer text only)
cot_response = assistant.chain_of_thought_reasoning(query, relevant_chunks)
print("CoT Response:\n")
print(cot_response)

# (Optional) save it
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out = cfg.results_analyses_dir / f"nb_cot_{ts}.txt"
out.write_text(cot_response, encoding="utf-8")
print("\nSaved →", out)


CoT Response:

Final Answer:
Supervised learning generally achieves higher accuracy and faster convergence by leveraging labeled data for explicit guidance, as demonstrated by improved FID scores (e.g., 17.2 baseline vs. lower values with optimized learning rates) and stable alignment of learned shifts [S4]. In contrast, unsupervised learning—while useful for structuring raw data (e.g., voxelizing dense point clouds from multi-view features) [S1]—often requires additional techniques like dynamic augmentation (e.g., motion-reversed videos) to mitigate artifacts and ensure robust representation [S3]. Supervised methods excel in tasks demanding precision (e.g., 3D reconstruction fidelity [S3]), whereas unsupervised approaches offer flexibility in exploratory or data-scarce scenarios but may suffer from slower convergence or instability without careful hyperparameter tuning [S4].

Saved → C:\Users\Windows\python\lesson\ai\capstone folder for ai\project_step_by_step\results\analyses\nb_cot_

In [26]:
from datetime import datetime
topic_q = "What are the current challenges in deep learning?"

# Run ReAct to produce a final answer
react_answer = assistant.react_research_workflow(topic_q, max_steps=4, search_k=4)
print("ReAct Final Answer:\n")
print(react_answer)

# (Optional) verify + improve the answer against context
ctx_hits = dp.find_similar_chunks(topic_q, top_k=4)
ctx_chunks = [h[1] for h in ctx_hits]
ver = assistant.verify_and_edit_answer(react_answer, topic_q, ctx_chunks)
print("\nVerified Final Answer:\n")
print(ver["final_answer"])

# Save both
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
(cfg.results_analyses_dir / f"nb_react_{ts}.txt").write_text(react_answer, encoding="utf-8")
(cfg.results_analyses_dir / f"nb_verified_{ts}.txt").write_text(ver["final_answer"], encoding="utf-8")


ReAct Final Answer:

Final Answer: Current challenges in deep learning include the **reliance on large-scale, multi-view real-world data** for 3D reconstruction, which is often unavailable [S4], and the **2D limitations of video diffusion models** that hinder applications in robotics, autonomous driving, and simulation [S4]. Additionally, **training instability** arises from hyperparameter sensitivity, such as learning rate disparities between lightweight conditioning networks and backbone models, creating "race conditions" that affect convergence and shift magnitude [S3]. Advances in **4D content generation** and **3D Gaussian Splatting (3DGS)** aim to address these gaps, but scalability and generalization remain open issues [S2][S4].

Verified Final Answer:

Final Answer: Current challenges in deep learning include the **reliance on large-scale, multi-view real-world data** for 3D reconstruction, which is often unavailable [S4], and the **2D limitations of video diffusion models** th

724